# A reusable timeseries annotation template

**Synthetic illustration — no experiment data or movement measurements are used here.**

This notebook demonstrates reusable Matplotlib labels: outbound/inbound regions, derived trial boundaries, raw Start Trial events, decision pokes, trial ends and outcomes. The helper reads no files and can annotate any existing timeseries axis. Movement analysis belongs in the figure notebooks; the artificial signals below only make the labels visible.

In [ ]:
from pathlib import Path
import sys

# Works from this notebook folder or any ancestor inside the project.
HERE = Path.cwd().resolve()
PROJECT_SRC = next(
    (candidate for parent in (HERE, *HERE.parents) for candidate in (parent, parent / "src")
     if (candidate / "movement_figures").is_dir()),
    None,
)
if PROJECT_SRC is None:
    raise RuntimeError("Open this notebook from the project checkout or one of its descendants.")
if str(PROJECT_SRC) not in sys.path:
    sys.path.insert(0, str(PROJECT_SRC))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from movement_figures.timeseries_template.annotations import (
    Annotations, Event, Region,
    QC_EVENT_STYLES, QC_OUTCOME_STYLES, QC_REGION_STYLES,
    annotate_qc_timeseries, draw_annotations, qc_annotations,
)

## Synthetic input in the Q_C table format

All timestamps are **absolute session-clock seconds**. Trials 1 and 2 reached the target zone. Trial 3 has no target trigger and ends as a Miss with `ChosenPort=-1`. Its phase spans will be omitted rather than inventing an inbound/outbound boundary.

The raw Start Trial events deliberately occur after the derived boundaries to show that these are different annotations.

In [ ]:
trials = pd.DataFrame({
    "session": ["synthetic_session"] * 3,
    "trial_index": [0, 1, 2],
    "start_time": [10.0, 20.0, 30.0],
    "end_time": [20.0, 30.0, 40.0],
    "outbound_start_time": [10.0, 20.0, 30.0],
    "outbound_end_time": [16.0, 26.0, np.nan],
    "inbound_start_time": [16.0, 26.0, np.nan],
    "inbound_end_time": [20.0, 30.0, 40.0],
    "outcome": ["Success", "Failure", "Miss"],
    "ChosenPort": [3, 7, -1],
})
events = pd.DataFrame({
    "Time": [10.4, 16.2, 19.8, 20.4, 26.2, 29.8, 30.4],
    "Event": [
        "Start Trial 1", "Await poke - Cue Tone ON", "NosePokes & CueTone OFF",
        "Start Trial 2", "Await poke - Cue Tone ON", "NosePokes & CueTone OFF",
        "Start Trial 3",
    ],
    "session": ["synthetic_session"] * 7,
}).set_index("Time")
t = np.linspace(9.0, 41.0, 1601)
signal = 0.5 + 0.25 * np.sin(t * 1.8) + 0.12 * np.cos(t * 4.5)
trials

## Default labels and a key

Each label appears once in the legend even when repeated across trials. Outcome markers sit at 96% of the axes height, independent of the plotted signal's units. A closing event is called a decision poke only when a valid chosen port was recorded and the outcome is not Miss.

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(t, signal, color="0.2", linewidth=1, label="Synthetic signal")
ax.set(xlabel="Session time (s)", ylabel="Synthetic amplitude (a.u.)",
       title="Synthetic example: default Q_C annotations", ylim=(-0.05, 1.15))
result = annotate_qc_timeseries(ax, trials, events, window=(9.0, 41.0))
ax.legend(handles=result.legend_handles, loc="upper center", bbox_to_anchor=(0.5, -0.2),
          ncol=4, fontsize=8)
fig.tight_layout()
plt.show()
for note in result.notes:
    print(note)

## Select, restyle and extend the template

Passing `None` uses a category's defaults; passing `{}` hides the category. A supplied mapping **replaces** that category, so include each kind you want. Copy an entry before changing it to preserve the shared defaults.

For Q_C regions choose `outbound` and/or `inbound`. Event keys `trial_start`, `poke` and `trial_end` refer to the trial table; other keys match raw event-name prefixes. The generic `Region` and `Event` records add arbitrary intervals and events without changing the Q_C adapter.

In [ ]:
base = qc_annotations(
    trials, events,
    region_styles={
        "inbound": {**QC_REGION_STYLES["inbound"], "color": "#c07a32", "alpha": 0.18},
    },
    event_styles={
        "Start Trial": {**QC_EVENT_STYLES["Start Trial"], "color": "#7551a8"},
        "poke": QC_EVENT_STYLES["poke"],
        "trial_end": QC_EVENT_STYLES["trial_end"],
    },
    outcome_styles={name: {**style, "height": 0.90} for name, style in QC_OUTCOME_STYLES.items()},
)
custom = Annotations(
    regions=base.regions + (Region(22.0, 24.0, "Example review interval", {"color": "#d5cb46", "alpha": 0.25}),),
    events=base.events + (Event(23.0, "Example sync pulse", {"color": "#126c7d", "marker": "D"}, height=0.12),),
    notes=base.notes,
)
fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(t, signal, color="0.2", linewidth=1, label="Synthetic signal")
ax.set(xlabel="Session time (s)", ylabel="Synthetic amplitude (a.u.)",
       title="Synthetic example: selected styles and custom annotations", ylim=(-0.05, 1.15))
result = draw_annotations(ax, custom, window=(9.0, 41.0))
ax.legend(handles=result.legend_handles, loc="upper center", bbox_to_anchor=(0.5, -0.2),
          ncol=4, fontsize=8)
fig.tight_layout()
plt.show()

## Reuse on multiple axes with relative time

Build the records once, then draw on each axis. `window` always uses absolute session time; the displayed x coordinate is `absolute time - time_offset`. Apply the same subtraction to the trace.

Plot windows include both boundaries `[start, end]`, so an entire-trial window includes the closing event and outcome marker. Adjacent panels may therefore each show an event on their shared boundary. This plotting convention does not determine how analysis samples are assigned to trials. Y limits are preserved, including when different panels use different units.

In [ ]:
time_offset = float(trials.iloc[0]["start_time"])
window = (10.0, 30.0)  # Includes both completed trials and the outcome at the right boundary.
shared = qc_annotations(trials, events)
fig, axes = plt.subplots(2, 1, figsize=(13, 6), sharex=True)
for ax, values, ylim, ylabel in zip(
    axes,
    (signal, 100 + 20 * signal),
    ((-0.05, 1.15), (95, 125)),
    ("Synthetic amplitude A", "Synthetic amplitude B"),
):
    ax.plot(t - time_offset, values, color="0.2", linewidth=1)
    ax.set(ylim=ylim, ylabel=ylabel)
    result = draw_annotations(ax, shared, window=window, time_offset=time_offset, legend=False)
axes[0].set_title("Synthetic example: shared labels on axes with different scales")
axes[-1].set_xlabel("Time from first derived trial boundary (s)")
fig.legend(handles=result.legend_handles, loc="lower center", ncol=4, fontsize=8)
fig.tight_layout(rect=(0, 0.12, 1, 1))
plt.show()

## Apply to a real notebook

Load the session once using the import template, select that session's trials and raw events, and plot the movement result first. Then call:

```python
result = annotate_qc_timeseries(
    ax, trials_one_session, events_one_session,
    window=(absolute_start, absolute_end),
)
for note in result.notes:
    print(note)
```

Do not mix sessions or clocks. If raw events are unavailable, pass `events=None`; trial-table labels still work. Missing chosen-port data produces a generic Trial end marker. Missing phase boundaries are reported through `result.notes`. Additional physical beam-break onsets are a separate data source and should be passed as explicit `Event` records when needed.